In [ ]:
##Organizei os arquivos csv como grafico1, 2, 3 e afins, porque precisava apenas dos dados
## para exportar para o editor e utilizar no desenvolvimento de gráficos


import pandas as pd

In [40]:
# Carregar o dataset
df = pd.read_csv("socials.csv")

# Exibir informações sobre colunas e tipos de dados
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Student_ID                   4500 non-null   str    
 1   Age                          4500 non-null   int64  
 2   Gender                       4500 non-null   str    
 3   Academic_Level               4500 non-null   str    
 4   Primary_Platform             4500 non-null   str    
 5   Daily_Usage_Hours            4500 non-null   float64
 6   Weekend_Extra_Hours          4500 non-null   float64
 7   Device_Type                  4500 non-null   str    
 8   Sleep_Duration_Hours         4500 non-null   float64
 9   Sleep_Quality_Score          4500 non-null   int64  
 10  Late_Night_Usage             4500 non-null   bool   
 11  Social_Comparison_Frequency  4500 non-null   str    
 12  Perceived_Stress_Score       4454 non-null   float64
 13  Mental_Health_Index          

In [ ]:
df = pd.read_csv("socials.csv")

amostra = df.sample(n=800, random_state=40).copy()

amostra.to_csv("amostra_800.csv", index=False)

print("Number of observations:", len(amostra))
print(amostra.head())

Number of observations: 800
       Student_ID  Age  Gender Academic_Level Primary_Platform  \
3317  STU_2027317   24    Male   Postgraduate        Instagram   
3189  STU_2027189   16    Male    High School         Snapchat   
4328  STU_2028328   18  Female  Undergraduate         Snapchat   
2358  STU_2026358   19    Male  Undergraduate         Snapchat   
302   STU_2024302   17  Female    High School          YouTube   

      Daily_Usage_Hours  Weekend_Extra_Hours Device_Type  \
3317                5.9                  2.4  Smartphone   
3189                4.9                  1.5  Smartphone   
4328                7.0                  1.3  Smartphone   
2358               10.8                  1.4  Smartphone   
302                 5.3                  1.8  Smartphone   

      Sleep_Duration_Hours  Sleep_Quality_Score  Late_Night_Usage  \
3317                   6.9                    2              True   
3189                   7.5                    2             False   
4328   

In [42]:
# 1 - Uso diário X Saúde Mental
grafico1 = amostra[
    ["Daily_Usage_Hours", "Mental_Health_Index"]
].dropna().rename(columns={
    "Daily_Usage_Hours": "X",
    "Mental_Health_Index": "Y"
}).sort_values("X")

grafico1.to_csv("grafico1.csv", index=False)

In [43]:
# 2 - Plataforma Principal
grafico2 = (
    amostra["Primary_Platform"]
    .dropna()
    .value_counts()
    .rename_axis("Plataforma")
    .reset_index(name="Frequencia")
)

grafico2["Porcentagem"] = (
    grafico2["Frequencia"] / grafico2["Frequencia"].sum() * 100
)

grafico2.to_csv("grafico2.csv", index=False)

In [44]:
dados = amostra[
    ["Daily_Usage_Hours", "Mental_Health_Index"]
].dropna()

pearson = dados["Daily_Usage_Hours"].corr(
    dados["Mental_Health_Index"],
    method="pearson"
)
print(pearson)

-0.8267829953884925


In [45]:
# 3. Qualidade do sono e tempo, baseado no uso até tarde

grafico3 = (
    amostra[
        [
            "Late_Night_Usage",
            "Sleep_Duration_Hours",
            "Sleep_Quality_Score"
        ]
    ]
    .dropna()
    .groupby("Late_Night_Usage")
    .agg(
        Average_Sleep_Duration=("Sleep_Duration_Hours", "mean"),
        Average_Sleep_Quality=("Sleep_Quality_Score", "mean"),
        Number_of_Students=("Sleep_Duration_Hours", "count"),
        Sleep_Duration_SD=("Sleep_Duration_Hours", "std"),
        Sleep_Quality_SD=("Sleep_Quality_Score", "std")
    )
    .reset_index()
    .rename(columns={
        "Late_Night_Usage": "Late_Night_Usage"
    })
)

grafico3.to_csv("grafico3.csv", index=False)

In [ ]:
#Outro gráfico de sono

dados = amostra["Late_Night_Usage"].dropna()

resultado = (
    dados.value_counts()
    .rename_axis("Uso_ate_tarde")
    .reset_index(name="Frequencia")
)

resultado["Porcentagem"] = (
    resultado["Frequencia"] / len(dados) * 100
).round(2)

resultado["Uso_ate_tarde"] = resultado["Uso_ate_tarde"].map({
    True: "Usam até tarde",
    False: "Não usam até tarde",
    "True": "Usam até tarde",
    "False": "Não usam até tarde"
})

resultado.to_csv(
    "grafico_sono_proporcoes.csv",
    index=False,
    encoding="utf-8-sig"
)

print(resultado.to_string(index=False))
print(f"\nTotal de respostas válidas: {len(dados)}")

     Uso_ate_tarde  Frequencia  Porcentagem
    Usam até tarde         437        54.62
Não usam até tarde         363        45.38

Total de respostas válidas: 800


In [47]:
#Ultimo Gráfico

dados = amostra[
    ["Daily_Usage_Hours", "Academic_Performance_GPA"]
].dropna().copy()

dados["Faixa_de_uso"] = pd.cut(
    dados["Daily_Usage_Hours"],
    bins=[-np.inf, 2, 4, 6, np.inf],
    labels=[
        "Até 2 horas",
        "Mais de 2 a 4 horas",
        "Mais de 4 a 6 horas",
        "Mais de 6 horas"
    ]
)

resultado = (
    dados.groupby("Faixa_de_uso", observed=False)
    ["Academic_Performance_GPA"]
    .mean()
    .reset_index()
)

resultado.columns = ["Faixa_de_uso", "GPA_medio"]
resultado["GPA_medio"] = resultado["GPA_medio"].round(3)

resultado.to_csv(
    "gpa_medio_por_faixa.csv",
    index=False,
    encoding="utf-8-sig"
)

print(resultado)

          Faixa_de_uso  GPA_medio
0          Até 2 horas      3.820
1  Mais de 2 a 4 horas      3.691
2  Mais de 4 a 6 horas      3.464
3      Mais de 6 horas      3.062
